# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

# Step 1: Install required packages

In [1]:
%pip install -U langchain langchain-openai langchain-community langchain-classic langchain-groq faiss-cpu tiktoken python-dotenv fastembed

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 1.1.14
    Uninstalling langchain-openai-1.1.14:
      Successfully uninstalled langchain-openai-1.1.14
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\AgrawalN\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Step 2: Import dependencies

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# Step 3: Set Groq API credentials
# Get a free API key from https://console.groq.com

In [3]:
# Load environment variables from the .env file in the same folder as this notebook
dotenv_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env")
load_dotenv(dotenv_path=dotenv_path, override=True)

# Verify the Groq key was loaded and show first/last 4 chars to confirm correct key
key = os.environ.get("GROQ_API_KEY")
if not key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")
print(f"Loaded from: {dotenv_path}")
print(f"Loaded key: {key[:8]}...{key[-4:]}")

Loaded from: c:\Nikhil\Work_Related\AI\MSAGI\AI\4-LLM Internals-PlanningSystems\Demos\Lesson_02_Designing_and_Deploying_Intelligent_Agents\Demo_01_Building_a_RAG_Powered_FAQ_Agent_with_Custom_Knowledge\.env
Loaded key: gsk_ZdJv...z8NF


# Step 4: Load and chunk your custom FAQ document


In [4]:
# Load the FAQ document and split it into chunks for embedding

# loading the txt  document file
# langchain supports many types of document formats, such as PDF, Word, HTML, etc. 
# You can use the appropriate loader for your document type. 
# In this example, we are using TextLoader for a plain text file.
loader = TextLoader("faq.txt")  # Ensure this file exists
documents = loader.load()

# look in text for by default \n\n characters paragraph breaks. 
# If it doesn't find the character, it will take the first 500 characters as a chunk
# chunk overlap means a certain characters can be in multiple chunks.
# if we want to change the separator, we can use the separator parameter 
# in the CharacterTextSplitter.
# For example, if we want to split by sentences, 
# we can use separator="\n\n" or separator="," depending on how the document is structured.
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

In [7]:
docs

[Document(metadata={'source': 'faq.txt'}, page_content='COMPANY FAQ - TechStore Customer Support\n\n========================================\nSHIPPING & DELIVERY\n========================================'),
 Document(metadata={'source': 'faq.txt'}, page_content='Q: What is your shipping policy?\nA: We offer multiple shipping options to meet your needs. Standard shipping takes 5-7 business days and costs $5.99. Express shipping takes 2-3 business days and costs $12.99. Premium overnight shipping is available for $24.99. Free standard shipping is available on all orders over $50. All orders are processed within 1-2 business days.'),
 Document(metadata={'source': 'faq.txt'}, page_content='Q: Do you ship internationally?\nA: Yes, we ship to over 150 countries worldwide. International shipping typically takes 10-21 business days depending on the destination. Shipping costs are calculated at checkout based on your location and order weight. Please note that customers are responsible for any 

# Step 5: Create vectorstore using local FastEmbed embeddings

In [8]:
# Embed document chunks using FastEmbed - a lightweight local embedding library
# Uses ONNX runtime instead of PyTorch, so no long-path issues on Windows
# Model is downloaded automatically (~50MB) on first run

embeddings = FastEmbedEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"  # fast, accurate, runs fully locally
)

# FAISS.from_documents() will embed and store all chunks in memory
vectorstore = FAISS.from_documents(docs, embeddings)

C:\Users\AgrawalN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
C:\Users\AgrawalN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\AgrawalN\AppData\Local\Temp\fastembed_cache\models--qdrant--bge-small-en-v1.5-onnx-q. Caching files will still work but in

# Step 6: Initialize the Groq LLM

In [9]:
# Initialize the Groq LLM (free API, no Azure needed)
# temperature=0 means deterministic output - always picks the most likely next word
# The LLM's job is to read the retrieved FAQ chunks and synthesize a coherent answer — 
# temperature=0 ensures it sticks to the facts rather than making things up.

llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # free, fast, high-quality open-source model
    temperature=0  # conservative, only sample most likely result
)

# Step 7: Create the RAG chain

In [10]:
# Create a RetrievalQA chain that uses the retriever and 
# LLM to answer queries with source context
# Here is our RAG Pipeline, connecting the vector store to the LLM.

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    # The retriever is what allows the chain to fetch relevant document 
    # chunks from our vector store based on the user's query.
    retriever=vectorstore.as_retriever(), 
    #tells the chain to return the source document chunks it retrieved along with the answer.
    return_source_documents=True 
)

# Step 8: Ask a question

In [14]:
# Send a question to the RAG chain and store the result

query = "What is our return policy?"
# Typically we would do this in the BE.
# We will have a FE to grab the prompt
# invoke() sends the query to the chain, which retrieves 
# relevant chunks from the vector store and generates an answer using the LLM.
result = qa_chain.invoke({"query": query})

In [15]:
result

{'query': 'What is our return policy?',
 'result': "We offer a 30-day return policy for all products, with some exceptions. Items must be in their original condition, unused, and in the original packaging. To initiate a return, log into your account, go to Order History, and select the return option. You'll receive a prepaid return shipping label via email. Once we receive and inspect your return, your refund will be processed within 5-7 business days. However, please note that certain items such as opened software, personal care items, earbuds, customized/personalized products, digital downloads, and gift cards are non-returnable.",
 'source_documents': [Document(id='bdd2a7c8-bb4c-4a8b-94b7-9514e7702120', metadata={'source': 'faq.txt'}, page_content="Q: What is your return policy?\nA: We offer a 30-day return policy for all products. Items must be in their original condition, unused, and in the original packaging. To initiate a return, log into your account, go to Order History, and s

# Step 9: Print results

In [16]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

Answer: We offer a 30-day return policy for all products, with some exceptions. Items must be in their original condition, unused, and in the original packaging. To initiate a return, log into your account, go to Order History, and select the return option. You'll receive a prepaid return shipping label via email. Once we receive and inspect your return, your refund will be processed within 5-7 business days. However, please note that certain items such as opened software, personal care items, earbuds, customized/personalized products, digital downloads, and gift cards are non-returnable.

--- Sources ---

Source 1:
Q: What is your return policy?
A: We offer a 30-day return policy for all products. Items must be in their original condition, unused, and in the original packaging. To initiate a return, log into your account, go to Order History, and select the return option. You'll receive a prepaid return shipping label via email. Once we receive and inspect your return, your refund wil

# Step 10: Interactive FAQ Q&A loop
Type your questions and get answers from the FAQ. Type **quit** or **exit** to stop.

In [ ]:
# Interactive loop: keep asking questions until the user types 'quit' or 'exit'
print("Welcome to the FAQ Assistant! Type 'quit' or 'exit' to stop.\n")

while True:
    query = input("Your question: ").strip()
    
    if query.lower() in ("quit", "exit"):
        print("Goodbye!")
        break
    
    if not query:
        print("Please enter a question.\n")
        continue
    
    result = qa_chain.invoke({"query": query})
    
    print(f"\nAnswer: {result['result']}")
    print("\n--- Sources ---")
    for i, doc in enumerate(result["source_documents"], 1):
        print(f"\nSource {i}:")
        print(doc.page_content)
    print("\n" + "=" * 50 + "\n")

Welcome to the FAQ Assistant! Type 'quit' or 'exit' to stop.

